# Grupo 7 - La Memoria de Pez
## Taller: Deep Learning Audit - El Rescate de HealthTech
**Universidad Privada del Norte - Escuela de Posgrado**

---

### Sabotaje asignado
El modelo tiene un **dataset pequeno** pero **no usa Regularizacion ni Weight Decay**, sufriendo un **sobreajuste (overfitting) violento**.

### Objetivo
1. Ejecutar el modelo saboteado y observar el overfitting.
2. Diagnosticar la causa raiz.
3. Aplicar la correccion: **Dropout + Weight Decay**.
4. Comparar resultados Antes vs Despues.

---
## PARTE 1: MODELO SABOTEADO (Sin Regularizacion)
---

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import make_circles
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)
np.random.seed(42)

### 1. Generacion de Datos
**SABOTAJE:** Usamos solo **100 muestras** (en vez de 1000) para simular un dataset medico pequeno.

In [ ]:
# =====================================================
# 1. GENERACION DE DATOS - DATASET PEQUENO (SABOTAJE)
# =====================================================
# SABOTAJE: Solo 100 muestras en vez de 1000
X_np, y_np = make_circles(n_samples=100, noise=0.05, factor=0.3, random_state=42)

# Split manual: 60 train, 40 test
indices = np.random.permutation(100)
train_idx, test_idx = indices[:60], indices[60:]

X_train_np, y_train_np = X_np[train_idx], y_np[train_idx]
X_test_np, y_test_np = X_np[test_idx], y_np[test_idx]

# Visualizacion
plt.figure(figsize=(6,6))
plt.scatter(X_np[y_np==0, 0], X_np[y_np==0, 1], color='red', label='Clase 0 (Sanos)')
plt.scatter(X_np[y_np==1, 0], X_np[y_np==1, 1], color='blue', label='Clase 1 (Enfermos)')
plt.title(f"Datos de HealthTech - Solo {len(X_np)} muestras (dataset pequeno)")
plt.legend()
plt.show()

# Conversion a Tensores
X_train = torch.from_numpy(X_train_np).float()
y_train = torch.from_numpy(y_train_np).float().unsqueeze(1)
X_test = torch.from_numpy(X_test_np).float()
y_test = torch.from_numpy(y_test_np).float().unsqueeze(1)

# Tambien mantenemos el dataset completo para graficar
X = torch.from_numpy(X_np).float()
y = torch.from_numpy(y_np).float().unsqueeze(1)

print(f'\nTrain: {len(X_train)} muestras | Test: {len(X_test)} muestras')
print(f'ADVERTENCIA: Dataset MUY pequeno - alto riesgo de overfitting')

### 2. Definicion de la Red (Saboteada)
Red **sobredimensionada** para 60 muestras, **sin Dropout ni regularizacion**.

In [ ]:
# =====================================================
# 2. DEFINICION DE LA RED - SABOTEADA
# =====================================================
class HealthNet(nn.Module):
    """Red sobredimensionada SIN regularizacion.
    SABOTAJE: No usa Dropout ni Weight Decay.
    Con 60 muestras y muchos parametros, memorizara todo."""
    def __init__(self, activation_type="relu"):
        super().__init__()
        # Red grande para tan pocos datos (sobredimensionada)
        self.layers = nn.Sequential(
            nn.Linear(2, 64),
            self._get_activation(activation_type),
            nn.Linear(64, 128),
            self._get_activation(activation_type),
            nn.Linear(128, 64),
            self._get_activation(activation_type),
            nn.Linear(64, 32),
            self._get_activation(activation_type),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def _get_activation(self, itype):
        if itype == "sigmoid": return nn.Sigmoid()
        if itype == "relu": return nn.ReLU()
        if itype == "leaky": return nn.LeakyReLU(0.01)

    def forward(self, x):
        return self.layers(x)

model_saboteado = HealthNet(activation_type="relu")
n_params = sum(p.numel() for p in model_saboteado.parameters())
print(f'Parametros del modelo: {n_params:,}')
print(f'Muestras de entrenamiento: {len(X_train)}')
print(f'Ratio parametros/muestras: {n_params/len(X_train):.0f}:1')
print(f'\nRatio >> 1 = El modelo puede MEMORIZAR todo el dataset')

### 3. Entrenamiento del Modelo Saboteado
Optimizador **sin weight_decay** (L2=0). El modelo memorizara los datos de entrenamiento.

In [ ]:
# =====================================================
# 3. BUCLE DE ENTRENAMIENTO - SABOTEADO
# =====================================================
def train_model(model, X_tr, y_tr, X_te, y_te, criterion, optimizer, epochs=1000):
    """Entrena y registra metricas de train y test."""
    history = {'train_loss': [], 'test_loss': [], 'train_acc': [], 'test_acc': []}

    for epoch in range(epochs):
        # --- Train ---
        model.train()
        outputs = model(X_tr)
        loss = criterion(outputs, y_tr)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_acc = ((outputs > 0.5).float() == y_tr).float().mean().item()

        # --- Test ---
        model.eval()
        with torch.no_grad():
            test_outputs = model(X_te)
            test_loss = criterion(test_outputs, y_te)
            test_acc = ((test_outputs > 0.5).float() == y_te).float().mean().item()

        history['train_loss'].append(loss.item())
        history['test_loss'].append(test_loss.item())
        history['train_acc'].append(train_acc)
        history['test_acc'].append(test_acc)

        if (epoch+1) % 200 == 0:
            print(f'Epoch [{epoch+1}/{epochs}] | '
                  f'Train Loss: {loss.item():.4f} | Test Loss: {test_loss.item():.4f} | '
                  f'Train Acc: {train_acc:.2%} | Test Acc: {test_acc:.2%}')

    return history

# SABOTAJE: Sin weight_decay (0.0) y sin momentum
criterion = nn.BCELoss()
optimizer_sab = optim.Adam(model_saboteado.parameters(), lr=0.01, weight_decay=0.0)  # SIN DECAY

print('=' * 70)
print('ENTRENANDO MODELO SABOTEADO (Sin Regularizacion ni Weight Decay)')
print('=' * 70)
history_sab = train_model(model_saboteado, X_train, y_train, X_test, y_test,
                          criterion, optimizer_sab, epochs=1000)

### 4. Frontera de Decision y Diagnostico

In [ ]:
# =====================================================
# 4. FRONTERA DE DECISION - SABOTEADO
# =====================================================
def plot_decision_boundary(model, X, y, title="Frontera de Decision"):
    x_min, x_max = X[:, 0].min() - 0.1, X[:, 0].max() + 0.1
    y_min, y_max = X[:, 1].min() - 0.1, X[:, 1].max() + 0.1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
    grid = torch.from_numpy(np.c_[xx.ravel(), yy.ravel()]).float()

    model.eval()
    with torch.no_grad():
        preds = model(grid).reshape(xx.shape)

    plt.contourf(xx, yy, preds.numpy(), alpha=0.3, cmap="RdYlBu")
    plt.scatter(X[:, 0], X[:, 1], c=y.squeeze().numpy(), edgecolors='k', cmap="RdYlBu")
    plt.title(title, fontsize=14, fontweight='bold')
    plt.show()

plot_decision_boundary(model_saboteado, X, y,
                       "SABOTEADO: Frontera sobreajustada (memoriza ruido)")

In [ ]:
# Graficar curvas de perdida y accuracy
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history_sab['train_loss'], label='Train Loss', color='blue')
axes[0].plot(history_sab['test_loss'], label='Test Loss', color='red')
axes[0].set_title('SABOTEADO - Curva de Perdida', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('BCE Loss')
axes[0].legend(fontsize=12)

axes[1].plot(history_sab['train_acc'], label='Train Acc', color='blue')
axes[1].plot(history_sab['test_acc'], label='Test Acc', color='red')
axes[1].set_title('SABOTEADO - Accuracy', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend(fontsize=12)
axes[1].set_ylim([0.4, 1.05])

plt.tight_layout()
plt.show()

gap_sab = history_sab['train_acc'][-1] - history_sab['test_acc'][-1]
print(f'\nDIAGNOSTICO:')
print(f'  Train Accuracy final: {history_sab["train_acc"][-1]:.2%}')
print(f'  Test Accuracy final:  {history_sab["test_acc"][-1]:.2%}')
print(f'  Brecha (Gap):         {gap_sab:.2%}')
print(f'\nSintoma: Train accuracy alta pero Test accuracy se estanca o baja.')
print(f'El modelo MEMORIZO los datos en vez de aprender patrones.')

---
## PARTE 2: MODELO CORREGIDO (Con Regularizacion)
---

### Interruptores que cambiamos:
1. **Weight Decay (L2=0.01)** en el optimizador Adam
2. **Dropout (p=0.3)** entre capas
3. **Reduccion de capacidad** de la red

In [ ]:
# =====================================================
# 5. MODELO CORREGIDO - CON REGULARIZACION
# =====================================================
class HealthNetCorregido(nn.Module):
    """Red con Dropout + arquitectura adecuada al dataset pequeno."""
    def __init__(self, activation_type="relu"):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(2, 16),
            self._get_activation(activation_type),
            nn.Dropout(0.3),       # REGULARIZACION: Dropout
            nn.Linear(16, 16),
            self._get_activation(activation_type),
            nn.Dropout(0.3),       # REGULARIZACION: Dropout
            nn.Linear(16, 1),
            nn.Sigmoid()
        )

    def _get_activation(self, itype):
        if itype == "sigmoid": return nn.Sigmoid()
        if itype == "relu": return nn.ReLU()
        if itype == "leaky": return nn.LeakyReLU(0.01)

    def forward(self, x):
        return self.layers(x)

model_corregido = HealthNetCorregido(activation_type="relu")
n_params_corr = sum(p.numel() for p in model_corregido.parameters())
print(f'Parametros saboteado:  {n_params:,}')
print(f'Parametros corregido:  {n_params_corr:,}')
print(f'Reduccion: {(1 - n_params_corr/n_params)*100:.1f}%')

In [ ]:
# CORRECCION: weight_decay=0.01 (L2 regularization)
optimizer_corr = optim.Adam(model_corregido.parameters(), lr=0.01, weight_decay=0.01)  # CON DECAY

print('=' * 70)
print('ENTRENANDO MODELO CORREGIDO (Con Dropout + Weight Decay)')
print('=' * 70)
history_corr = train_model(model_corregido, X_train, y_train, X_test, y_test,
                           criterion, optimizer_corr, epochs=1000)

In [ ]:
# Frontera de decision corregida
plot_decision_boundary(model_corregido, X, y,
                       "CORREGIDO: Frontera suave (generaliza)")

---
## PARTE 3: COMPARACION ANTES vs DESPUES
---

In [ ]:
# =====================================================
# 6. COMPARACION: ANTES vs DESPUES
# =====================================================
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Fila 1: SABOTEADO
axes[0, 0].plot(history_sab['train_loss'], label='Train', color='blue')
axes[0, 0].plot(history_sab['test_loss'], label='Test', color='red')
axes[0, 0].set_title('SABOTEADO - Perdida (Loss)', fontsize=13, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].set_ylabel('Loss')

axes[0, 1].plot(history_sab['train_acc'], label='Train', color='blue')
axes[0, 1].plot(history_sab['test_acc'], label='Test', color='red')
axes[0, 1].set_title('SABOTEADO - Accuracy', fontsize=13, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].set_ylim([0.4, 1.05])

# Fila 2: CORREGIDO
axes[1, 0].plot(history_corr['train_loss'], label='Train', color='blue')
axes[1, 0].plot(history_corr['test_loss'], label='Test', color='green')
axes[1, 0].set_title('CORREGIDO - Perdida (Loss)', fontsize=13, fontweight='bold')
axes[1, 0].legend()
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Loss')

axes[1, 1].plot(history_corr['train_acc'], label='Train', color='blue')
axes[1, 1].plot(history_corr['test_acc'], label='Test', color='green')
axes[1, 1].set_title('CORREGIDO - Accuracy', fontsize=13, fontweight='bold')
axes[1, 1].legend()
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Accuracy')
axes[1, 1].set_ylim([0.4, 1.05])

plt.suptitle('ANTES vs DESPUES - Grupo 7: La Memoria de Pez',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Fronteras de decision lado a lado
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

x_min, x_max = X[:, 0].min() - 0.1, X[:, 0].max() + 0.1
y_min, y_max = X[:, 1].min() - 0.1, X[:, 1].max() + 0.1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
grid = torch.from_numpy(np.c_[xx.ravel(), yy.ravel()]).float()

for i, (mod, title) in enumerate([
    (model_saboteado, 'SABOTEADO\n(frontera irregular, memoriza)'),
    (model_corregido, 'CORREGIDO\n(frontera suave, generaliza)')
]):
    mod.eval()
    with torch.no_grad():
        preds = mod(grid).reshape(xx.shape)
    axes[i].contourf(xx, yy, preds.numpy(), alpha=0.3, cmap="RdYlBu")
    axes[i].scatter(X[:, 0], X[:, 1], c=y.squeeze().numpy(), edgecolors='k', cmap="RdYlBu")
    axes[i].set_title(title, fontsize=13, fontweight='bold')

plt.suptitle('Frontera de Decision: Antes vs Despues', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Comparacion de distribucion de pesos
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

weights_sab = []
for name, param in model_saboteado.named_parameters():
    if 'weight' in name:
        weights_sab.extend(param.data.cpu().numpy().flatten())

weights_corr = []
for name, param in model_corregido.named_parameters():
    if 'weight' in name:
        weights_corr.extend(param.data.cpu().numpy().flatten())

axes[0].hist(weights_sab, bins=60, color='red', alpha=0.7, edgecolor='black')
axes[0].set_title('Pesos - Sin Regularizacion', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Valor del peso')
axes[0].axvline(x=0, color='black', linestyle='--')

axes[1].hist(weights_corr, bins=60, color='green', alpha=0.7, edgecolor='black')
axes[1].set_title('Pesos - Con Regularizacion', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Valor del peso')
axes[1].axvline(x=0, color='black', linestyle='--')

plt.tight_layout()
plt.show()

print(f'Pesos saboteado  - Std: {np.std(weights_sab):.4f}, Max |w|: {np.max(np.abs(weights_sab)):.4f}')
print(f'Pesos corregido  - Std: {np.std(weights_corr):.4f}, Max |w|: {np.max(np.abs(weights_corr)):.4f}')

In [ ]:
# =====================================================
# 7. TABLA RESUMEN FINAL
# =====================================================
gap_sab = history_sab['train_acc'][-1] - history_sab['test_acc'][-1]
gap_corr = history_corr['train_acc'][-1] - history_corr['test_acc'][-1]

print('\n' + '=' * 65)
print('           RESUMEN DE AUDITORIA - GRUPO 7')
print('=' * 65)
print(f'{"Metrica":<25} {"Saboteado":>18} {"Corregido":>18}')
print('-' * 65)
print(f'{"Train Accuracy":<25} {history_sab["train_acc"][-1]:>17.2%} {history_corr["train_acc"][-1]:>17.2%}')
print(f'{"Test Accuracy":<25} {history_sab["test_acc"][-1]:>17.2%} {history_corr["test_acc"][-1]:>17.2%}')
print(f'{"Gap (Train-Test)":<25} {gap_sab:>17.2%} {gap_corr:>17.2%}')
print(f'{"Train Loss":<25} {history_sab["train_loss"][-1]:>18.4f} {history_corr["train_loss"][-1]:>18.4f}')
print(f'{"Test Loss":<25} {history_sab["test_loss"][-1]:>18.4f} {history_corr["test_loss"][-1]:>18.4f}')
print(f'{"Parametros":<25} {n_params:>18,} {n_params_corr:>18,}')
print(f'{"Weight Decay":<25} {"0.0 (OFF)":>18} {"0.01 (ON)":>18}')
print(f'{"Dropout":<25} {"NO":>18} {"p=0.3":>18}')
print('=' * 65)

print(f'\nCONCLUSIONES:')
print(f'  1. DIAGNOSTICO: El modelo memorizaba el dataset de entrenamiento.')
print(f'     La perdida no bajaba de forma estable en test.')
print(f'  2. INTERRUPTOR: Cambiamos Weight Decay de 0 a 0.01 y agregamos Dropout.')
print(f'  3. RESULTADO: La brecha train-test se redujo de {gap_sab:.2%} a {gap_corr:.2%}.')
print(f'     El modelo ahora GENERALIZA en vez de memorizar.')

---
## Bonus Matematico: Efecto de L2 en el Gradiente

**Sin Weight Decay:**
$$\theta_{t+1} = \theta_t - \eta \nabla L(\theta_t)$$

**Con Weight Decay (L2):**
$$\theta_{t+1} = \theta_t - \eta (\nabla L(\theta_t) + \lambda \theta_t) = (1 - \eta\lambda)\theta_t - \eta \nabla L(\theta_t)$$

El termino $(1 - \eta\lambda)$ **encoge los pesos** en cada paso, penalizando pesos grandes que memorizan ruido.

**Dropout:** En cada forward pass, se anulan neuronas al azar con probabilidad $p$:
$$h_i = \begin{cases} 0 & \text{con probabilidad } p \\ \frac{x_i}{1-p} & \text{con probabilidad } 1-p \end{cases}$$

Esto fuerza a la red a **no depender de neuronas individuales**, distribuyendo el aprendizaje.

### Grafo de computacion:
$$L_{total} = L_{BCE}(\hat{y}, y) + \lambda \sum_i ||w_i||^2$$

El termino de regularizacion agrega un gradiente $2\lambda w_i$ que fluye hacia atras por todas las capas, controlando la magnitud de cada peso.